# LLM uncertainty test — free, on Google Colab

This notebook downloads an **open-weight model from Hugging Face**, runs it on Colab's **free GPU**, asks it your
multiple-choice questions, and records the probability behind every answer, then asks the model how confident it is (0–100) so the two can be compared. No API key and no spending.

Models available (pick one in **Settings**):

| Model | From | Notes |
|---|---|---|
| **Gemma 4 12B** | Google (built from Gemini 3 research) | Default. 4-bit, fits the free T4 |
| **Gemma 4 E4B** | Google | Smaller and faster, weaker. Good for quick tests |
| **gpt-oss-20b** | OpenAI | Experimental on a T4: if it can't load, it's skipped and logged |

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.* Then run the cells top to bottom (Shift+Enter).

In [47]:
#@title 1. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → T4 GPU, then run this cell again."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}, {p.total_memory / 1e9:.1f} GB")

GPU: Tesla T4, 15.6 GB


In [48]:
#@title 2. Get the code { display-mode: "form" }
#@markdown Path to `llm-uncertainty.zip` inside My Drive. Put the zip in your own Drive folder and set the path here. Clear it to use REPO_URL, or clear both to upload `llm-uncertainty.zip` from your computer.
CODE_ZIP_ON_DRIVE = "llm-uncertainty/llm-uncertainty.zip"  #@param {type:"string"}
REPO_URL = ""  #@param {type:"string"}
import os, zipfile, shutil
os.chdir("/content")
if CODE_ZIP_ON_DRIVE.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    zpath = os.path.join("/content/drive/MyDrive", CODE_ZIP_ON_DRIVE.strip())
    assert os.path.exists(zpath), (f"Can't find {zpath}. Check the path, or clear CODE_ZIP_ON_DRIVE "
                                   "to upload the zip from your computer instead.")
    shutil.rmtree("/content/llm-uncertainty", ignore_errors=True)   # always start from the latest zip
    zipfile.ZipFile(zpath).extractall("/content")
    print("Code taken from Drive:", zpath)
elif os.path.isdir("/content/llm-uncertainty"):
    print("Code already here, skipping.")
elif REPO_URL.strip():
    !git clone -q {REPO_URL.strip()} llm-uncertainty
else:
    from google.colab import files
    print("Choose llm-uncertainty.zip:")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
assert os.path.isdir("/content/llm-uncertainty"), "The zip should contain a folder called llm-uncertainty."
%cd /content/llm-uncertainty


In [49]:
#@title 3. Install packages (about 1 minute)
!pip install -q -r requirements-hf.txt
import transformers; print("transformers", transformers.__version__)

# T4 fix: on GPUs without bfloat16 (e.g. the free T4) llmtest keeps the non-quantised layers in
# float32, and Gemma 4's large embedding tables then don't fit in GPU memory ("Some modules are
# dispatched on the CPU or the disk"). Load those layers in float16 instead.
import re, torch
if torch.cuda.get_device_capability() < (8, 0):
    _cfg = open("config.colab.toml").read()
    _fixed = re.sub(r'(?m)^(quantization = "4bit"\n)(?!dtype)', r'\1dtype = "float16"\n', _cfg)
    if _fixed != _cfg:
        open("config.colab.toml", "w").write(_fixed)
        print("T4 detected: 4-bit models will load with dtype = float16")


transformers 5.23.0


In [ ]:
#@title 4. Settings { display-mode: "form" }
MODEL = "Gemma 4 E4B"  #@param ["Gemma 4 12B", "Gemma 4 E4B", "gpt-oss-20b", "All three"]
#@markdown **Questions from Google Drive** (My Drive ▸ folder below). Leave `DRIVE_FILE` blank to use the newest .csv in the folder.
QUESTIONS_FROM_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "llm-uncertainty"  #@param {type:"string"}
DRIVE_FILE = ""  #@param {type:"string"}
#@markdown Run only the first N questions (0 = all). Use 5 for a quick test.
FIRST_N_QUESTIONS = 0  #@param {type:"integer"}
#@markdown Used only when QUESTIONS_FROM_DRIVE is off:
QUESTIONS = "questions/example.csv"  #@param {type:"string"}
UPLOAD_MY_OWN_QUESTIONS = False  #@param {type:"boolean"}
RUN_NAME = "colab-run-2"  #@param {type:"string"}
#@markdown Saving to Google Drive keeps results if Colab disconnects. Re-run with the same RUN_NAME to resume.
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}

import glob
import pandas as pd

if QUESTIONS_FROM_DRIVE or SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Map the vets' sheet layout onto the columns llmtest expects (id, question, A, B, C, D, E, answer).
COLUMN_MAP = {"question_id": "id", "option_a": "A", "option_b": "B", "option_c": "C",
              "option_d": "D", "option_e": "E", "correct_option": "answer", "correct_answer": "answer"}

def drive_questions(folder, filename):
    base = os.path.join("/content/drive/MyDrive", folder)
    if not os.path.isdir(base):
        near = [d for d in os.listdir("/content/drive/MyDrive") if folder.lower()[:12] in d.lower()]
        raise FileNotFoundError(f"No folder '{folder}' in My Drive. Similar names: {near or 'none'}. "
                                "If it is under 'Shared with me', right-click it in Drive ▸ Organise ▸ Add shortcut to My Drive.")
    if filename.strip():
        return os.path.join(base, filename.strip())
    csvs = sorted(glob.glob(os.path.join(base, "*.csv")), key=os.path.getmtime)
    if not csvs:
        raise FileNotFoundError(f"No .csv files in My Drive/{folder}. In Google Sheets use File ▸ Download ▸ CSV and put it there, or set DRIVE_FILE.")
    return csvs[-1]

if QUESTIONS_FROM_DRIVE:
    src = drive_questions(DRIVE_FOLDER, DRIVE_FILE)
    qdf = pd.read_csv(src)
    qdf.columns = [c.strip() for c in qdf.columns]
    qdf = qdf.rename(columns={c: COLUMN_MAP.get(c.lower(), c) for c in qdf.columns})
    missing = [c for c in ("id", "question", "A", "B", "answer") if c not in qdf.columns]
    assert not missing, f"Questions file is missing columns {missing}. Found: {list(qdf.columns)}"
    qdf = qdf.dropna(subset=["question"])
    qdf["answer"] = qdf["answer"].astype(str).str.strip().str.upper().replace({"NAN": ""})
    if FIRST_N_QUESTIONS > 0:
        qdf = qdf.head(FIRST_N_QUESTIONS)
    os.makedirs("questions", exist_ok=True)
    QUESTIONS = "questions/from_drive.csv"
    qdf.to_csv(QUESTIONS, index=False)
    print(f"Loaded {len(qdf)} questions from {src}")
    print("Options per question:", [c for c in "ABCDEFGH" if c in qdf.columns])
elif UPLOAD_MY_OWN_QUESTIONS:
    from google.colab import files
    print("Choose your questions .csv or .jsonl (columns: id, question, A, B, C, D, answer):")
    up = files.upload()
    name = next(iter(up))
    os.makedirs("questions", exist_ok=True)
    os.replace(name, f"questions/{name}")
    QUESTIONS = f"questions/{name}"

if SAVE_TO_GOOGLE_DRIVE:
    OUT_DIR = "/content/drive/MyDrive/llm-uncertainty-results"
else:
    OUT_DIR = "/content/llm-uncertainty/results"
os.makedirs(OUT_DIR, exist_ok=True)
MODELS_ARG = "" if MODEL == "All three" else f'--models "{MODEL}"'
print(f"Questions: {QUESTIONS}\nModel(s): {MODEL}\nResults: {OUT_DIR}/{RUN_NAME}")


In [51]:
#@title 5. Run the test
#@markdown The first run downloads the model (Gemma 4 12B is ~24 GB, a few minutes). Then a few seconds per question.
!python -m llmtest run -c config.colab.toml -q "{QUESTIONS}" --name "{RUN_NAME}" --out "{OUT_DIR}" {MODELS_ARG}

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/llm-uncertainty/llmtest/__main__.py", line 127, in <module>
    main()
    ~~~~^^
  File "/content/llm-uncertainty/llmtest/__main__.py", line 123, in main
    args.func(args)
    ~~~~~~~~~^^^^^^
  File "/content/llm-uncertainty/llmtest/__main__.py", line 35, in cmd_run
    questions = load_questions(qpath)
  File "/content/llm-uncertainty/llmtest/questions.py", line 52, in load_questions
    raise ValueError("questions file must be .csv or .jsonl")
ValueError: questions file must be .csv or .jsonl


In [52]:
#@title 6. Show the report
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT_DIR}/{RUN_NAME}/report.md").read()))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/llm-uncertainty-results/colab-run-2/report.md'

In [ ]:
#@title 7. Explore the data
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f"{OUT_DIR}/{RUN_NAME}/summary.csv")
df = df[df["status"] == "ok"]
cols = ["model_label", "question_id", "answer", "correct_answer", "is_correct", "p_answer",
        "stated_conf", "stated_conf_raw", "probit", "entropy_bits", "margin"]
display(df[[c for c in cols if c in df.columns]])

graded = df.dropna(subset=["p_answer", "is_correct"])
if len(graded):
    has_stated = "stated_conf" in graded.columns and graded["stated_conf"].notna().any()
    fig, axes = plt.subplots(1, 2 if has_stated else 1, figsize=(11 if has_stated else 6, 3.8))
    ax = axes[0] if has_stated else axes
    for ok, label in [(True, "correct"), (False, "wrong")]:
        ax.hist(graded.loc[graded.is_correct == ok, "p_answer"], bins=10, range=(0, 1), alpha=0.6, label=label)
    ax.set_xlabel("token confidence in chosen answer (p)"); ax.set_ylabel("questions"); ax.legend()
    ax.set_title("Is the model more confident when it's right?")
    if has_stated:
        ax = axes[1]
        s = graded.dropna(subset=["stated_conf"])
        for ok, label, mk in [(True, "correct", "o"), (False, "wrong", "x")]:
            sub = s[s.is_correct == ok]
            ax.scatter(sub["p_answer"], sub["stated_conf"], marker=mk, alpha=0.7, label=label)
        ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
        ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
        ax.set_xlabel("token confidence (p)"); ax.set_ylabel("stated confidence (asked 0-100)")
        ax.set_title("What it says vs what its probabilities say"); ax.legend()
    plt.tight_layout(); plt.show()


In [ ]:
#@title 8. Download the results (report.md, summary.csv, raw.jsonl, run_meta.json)
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", f"{OUT_DIR}/{RUN_NAME}")
files.download(zip_path)

## Tips

- **Resume after a disconnect:** with *Save to Google Drive* on, run cells 1–5 again with the same `RUN_NAME`. Answers already saved are skipped.
- **New experiment:** change `RUN_NAME` (and the questions file). Each run gets its own folder.
- **Your own questions:** a CSV with columns `id, question, A, B, C, D, answer` (`answer` can be blank). Tick *Upload my own questions*.
- **Change model settings** in `config.colab.toml` (open it from the file panel on the left). For example, set gpt-oss to answer without reasoning.
- **Free Colab limits:** GPU sessions can disconnect after long idle periods or a few hours, and a GPU isn't always available. Resuming covers this.
- **Questions from Drive:** put the questions CSV in your Drive folder (set `DRIVE_FOLDER`, default *My Drive ▸ llm-uncertainty*). The newest CSV there is used unless you set `DRIVE_FILE`. Column names like `question_id, option_a … option_e, correct_option` are converted automatically.
- **Code zip:** cell 2 takes `llm-uncertainty.zip` from your Drive folder (`CODE_ZIP_ON_DRIVE`) each time, so replacing that file is all it takes to update the code.
- **Several models:** pick *All three* in Settings. They run one after another (one GPU), into one `summary.csv`.
